# ASD Screening — Model Integration Verification

This notebook verifies that the saved XGBoost pipeline used by the FastAPI backend produces the same predictions as the original training notebook. It walks through model loading, preprocessing verification, a sample prediction, and a live check against the API layer.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd

# Make the backend package importable when running from the repo root.
BACKEND_DIR = Path.cwd().parent / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("Backend directory:", BACKEND_DIR)
print("pandas version:", pd.__version__)

## 1. Load the trained pipeline and metadata

The model file contains the full training pipeline (preprocessor -> SMOTE -> classifier), so a single `predict()` call on raw data applies all the same transformations used during training.

In [ ]:
model = joblib.load(BACKEND_DIR / "artifacts" / "asd_best_model.joblib")
metadata = joblib.load(BACKEND_DIR / "artifacts" / "asd_model_metadata.joblib")

print("Pipeline steps:", list(model.named_steps.keys()))
print("Best model:", metadata["best_model_name"])
print("Number of features:", len(metadata["features"]))

## 2. Inspect the preprocessing pipeline

Confirm the ColumnTransformer contains a numerical branch (StandardScaler) and a categorical branch (OneHotEncoder), and that the encoder learned the expected categories.

In [ ]:
preprocessor = model.named_steps["preprocessor"]

for name, transformer, columns in preprocessor.transformers_:
    print(f"Branch: {name}")
    print(f"  Type: {type(transformer).__name__}")
    print(f"  Columns ({len(columns)}): {columns}")
    if name == "cat":
        for col, cats in zip(columns, transformer.categories_):
            print(f"  Categories for '{col}': {len(cats)} values")

## 3. Sample prediction (raw input)

This is the exact patient example from the training notebook. The pipeline handles all encoding internally, so the prediction should match the notebook's output.

In [ ]:
new_patient = pd.DataFrame([{
    "A1_Score": 1, "A2_Score": 1, "A3_Score": 0, "A4_Score": 1, "A5_Score": 1,
    "A6_Score": 0, "A7_Score": 1, "A8_Score": 1, "A9_Score": 0, "A10_Score": 1,
    "age": 27, "gender": 1, "ethnicity": "White-European", "jundice": 0,
    "austim": 0, "contry_of_res": "United States", "used_app_before": 0,
    "relation": "Self",
}])

prediction = model.predict(new_patient)[0]
probabilities = model.predict_proba(new_patient)[0]

print("Prediction:", prediction)
print("Probabilities (no ASD, ASD):", probabilities)
print("Label:", "ASD Detected" if prediction == 1 else "No ASD Detected")
print(f"Confidence: {probabilities[prediction]:.4f}")

## 4. Verify the backend service produces the same result

The prediction service in `backend/app` builds the DataFrame in the exact training order, so it must return the same values we got above.

In [ ]:
from app.models.schemas import ScreeningInput
from app.services.prediction_service import predict

payload = {
    "A1_Score": 1, "A2_Score": 1, "A3_Score": 0, "A4_Score": 1, "A5_Score": 1,
    "A6_Score": 0, "A7_Score": 1, "A8_Score": 1, "A9_Score": 0, "A10_Score": 1,
    "age": 27, "gender": 1, "ethnicity": "White-European",
    "contry_of_res": "United States", "jundice": 0, "austim": 0,
    "used_app_before": 0, "relation": "Self",
}

result = predict(ScreeningInput(**payload))
print(json.dumps(result, indent=2))

## 5. Consistency check

Assert that the backend service matches the raw pipeline output exactly.

In [ ]:
assert result["prediction"] == int(prediction), "Prediction mismatch!"
assert abs(result["probability"]["asd"] - float(probabilities[1])) < 1e-4, "Probability mismatch!"
print("✓ The backend prediction matches the trained model exactly.")

## 6. Test the full API layer

Use FastAPI's TestClient to exercise the real HTTP endpoints without starting a server.

In [ ]:
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)

health = client.get("/health")
print("GET /health ->", health.status_code, health.json())

info = client.get("/api/v1/model-info")
print("GET /api/v1/model-info ->", info.status_code)
print("  best_model =", info.json()["best_model_name"])
print("  features   =", len(info.json()["features"]))

response = client.post("/api/v1/predict", json=payload)
print("POST /api/v1/predict ->", response.status_code)
print(json.dumps(response.json(), indent=2))

## 7. Invalid input handling

The API should reject unknown categorical values and out-of-range scores with a 422 status.

In [ ]:
bad_ethnicity = {**payload, "ethnicity": "Not-A-Real-Ethnicity"}
resp1 = client.post("/api/v1/predict", json=bad_ethnicity)
print("Unknown ethnicity ->", resp1.status_code, "(expected 422)")

bad_score = {**payload, "A1_Score": 5}
resp2 = client.post("/api/v1/predict", json=bad_score)
print("Out-of-range score ->", resp2.status_code, "(expected 422)")

## Summary

- The saved pipeline applies the same preprocessing as training, so raw user input can be passed straight to `predict`.
- The backend prediction service and HTTP API produce results identical to the training notebook.
- Validation rejects invalid categorical values and out-of-range scores with proper HTTP error codes.

The application is ready to run locally with Docker or deploy to Vercel (frontend) and Render (backend).